In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import amp_pd_peptide
import numpy as np
from sklearn.linear_model import LinearRegression

In [ ]:
train = pd.read_csv("/kaggle/input/amp-parkinsons-disease-progression-prediction/train_clinical_data.csv")

In [ ]:
train= train.drop(columns= {'upd23b_clinical_state_on_medication'})
train.head()

In [ ]:
#train=train.apply(np.int64)

In [ ]:

train['updrs_1']= train['updrs_1'].fillna(train['updrs_1'].median())
train['updrs_2']= train['updrs_2'].fillna(train['updrs_2'].median())
train['updrs_3']= train['updrs_3'].fillna(0.0)


In [ ]:
train['updrs_3'].mode()

In [ ]:
train.info()

In [ ]:
train_df= train.dropna(subset=['updrs_4'])
test_df= train[train['updrs_4'].isna()]

In [ ]:
X_train= train_df[['updrs_1', 'updrs_3']]
y_train= train_df['updrs_4']

X_test=test_df[['updrs_1', 'updrs_3']] 

In [ ]:
my_model = LinearRegression()
my_model.fit(X_train, y_train)
y_pred= my_model.predict(X_test)

In [ ]:
test_df['updrs_4']= y_pred
#test_df

In [ ]:
train= [train_df, test_df]
train=pd.concat(train)
#display(train)

In [ ]:
model = {}
target = ["updrs_1", "updrs_2", "updrs_3", "updrs_4"]

for u in target:
    

    # Train data
    X = train['visit_month']
    y = train[u]
    
        
    trained = LinearRegression().fit(X.values.reshape(-1, 1), y)
    
    # Save model
    model[u] = trained

In [ ]:
def get_predictions(my_train, pro, model):

    # Forecast
    my_train = my_train.fillna(0)
    
    for u in target:
        
        # Here is where we will save the final results
        my_train['result_' + str(u)] = 0
  
        # Predict    
        X = my_train["visit_month"]
        
        
        my_train['result_' + str(u)] = np.ceil(model[u].predict(X.values.reshape(-1, 1)))
            
     # Format for final submission
    result = pd.DataFrame()

    for m in [0, 6, 12, 24]:
        for u in [1, 2, 3, 4]:

            temp = my_train[["visit_id", "result_updrs_" + str(u)]]
            temp["prediction_id"] = temp["visit_id"] + "_updrs_" + str(u) + "_plus_" + str(m) + "_months"
            temp["rating"] = temp["result_updrs_" + str(u)]
            temp = temp [['prediction_id', 'rating']]

            result = result.append(temp)            
    result = result.drop_duplicates(subset=['prediction_id', 'rating'])

    return result

In [ ]:
env = amp_pd_peptide.make_env()   # initialize the environment
iter_test = env.iter_test()  

In [ ]:
for (test, test_peptides, test_proteins, sample_submission) in iter_test:
        
    result = get_predictions(test, test_proteins, model)

    env.predict(result)   # register your predictions